# 01 — CiteCheck: graph-grounded relational verification (paper contribution)

Verifies each relational claim `(A, relation, B)` from the relational benchmark against the
**citation graph** — the failure mode text verifiers can't check. **Embedding-free** (does NOT
need notebook 03); builds a *focused* graph over only the benchmark's relational papers via the
S2 batch API. Output: `citecheck_results.json` (consumed by `02_relational_finding`).

Run on Kaggle: attach the dataset with `scireqa_full_relational_v1.1.jsonl`, Internet ON,
`SEMANTIC_SCHOLAR_API_KEY` in secrets. CPU is fine.

**Scope (v1):** structural verification (citation direction + shared references + year order).
Citation-*intent* classification (SciCite) is a planned enhancement. **Circularity note:** the
graph is the *predictor*; the *gold* labels for the finding must be human (notebook nb2), not the graph.

**Entity-resolution fix (applied offline, outside this notebook):** the benchmark's
`relational.paper_a`/`paper_b` fields used to be resolved via fuzzy title matching only, which
failed on ~51-56% of genuinely relational claims -- not because the citation graph was missing
those papers (manual audit found 0% of *resolved* pairs were absent from the graph), but because
the matcher never checked (a) the `[N]` bracket numbers rationales use to reference
`ground_truth_papers` positionally, or (b) "the query paper" as a valid participant. Re-running
mention resolution with both signals raised resolution from 43.9% to 83.4% of the 26,245
auto-detected relational claims (11,517 -> 21,897 checkable tuples) with zero changes to
MiniCheck verdicts/scores. `scireqa_full_relational_v1.1.jsonl` in this dataset already has the
corrected `paper_a`/`paper_b` fields -- **re-running this notebook now will fetch ~6,166 newly
resolved papers via S2** that weren't in `citecheck_refs_cache.json` before (the old matcher
never surfaced them, so they were never fetched). Re-upload the updated benchmark file to the
Kaggle dataset before running.

In [4]:
# == 1. Imports + locate relational benchmark + S2 key ========================
import json, os, re, sys, time, requests
from pathlib import Path
from collections import Counter, defaultdict

def _find(name):
    for root in ["/kaggle/input", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits: return hits[0]
    return None

BENCH = _find("scireqa_full_relational_v1.1.jsonl")
assert BENCH, "attach the dataset containing scireqa_full_relational_v1.1.jsonl"
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

S2_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    S2_KEY = UserSecretsClient().get_secret("SEMANTIC_SCHOLAR_API_KEY") or S2_KEY
except Exception:
    pass
S2_BASE = "https://api.semanticscholar.org/graph/v1"
HDRS = {"x-api-key": S2_KEY} if S2_KEY else {}
print("benchmark:", BENCH, "| S2 key:", bool(S2_KEY))

bench = [json.loads(l) for l in open(BENCH, encoding="utf-8") if l.strip()]
print("instances:", len(bench))

benchmark: /kaggle/input/datasets/joytuntonny/rationale-subset/scireqa_full_relational_v1.1.jsonl | S2 key: True
instances: 10000


In [5]:
# == 2. Collect relational (A, relation, B) tuples + the papers involved =======
ARXIV_RE = re.compile(r"^\d{4}\.\d{4,5}")
def s2_id(pid):
    pid = str(pid)
    return "ARXIV:" + pid if ARXIV_RE.match(pid) else pid     # arXiv ids need the prefix

tuples = []   # (inst_id, claim_id, A_raw, relation, B_raw)
papers = set()
for inst in bench:
    q = inst.get("query_paper_id")
    for lbl in inst.get("hallucination_labels", []):
        r = lbl.get("relational")
        if r and r.get("paper_a") and r.get("paper_b"):
            a, b = r["paper_a"], r["paper_b"]
            tuples.append((inst["id"], lbl["claim_id"], a, r["relation"], b))
            papers.add(a); papers.add(b)
    if q: papers.add(q)
print(f"relational tuples (both papers): {len(tuples)}")
print(f"unique papers to resolve: {len(papers)}")
print("relation types:", dict(Counter(t[3] for t in tuples)))

relational tuples (both papers): 11517
unique papers to resolve: 21156
relation types: {'builds_on': 6183, 'shares_topic': 2857, 'compares': 2394, 'supersedes': 83}


In [6]:
# == 3. Build a focused citation graph (S2 batch: references + year; CACHED) ====
from tqdm.auto import tqdm
CACHE_PATH = OUT_DIR / "citecheck_refs_cache.json"
_cache = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}

def _s2_batch(ids, fields, chunk=100):
    res = {}
    for i in tqdm(range(0, len(ids), chunk), desc="S2 batch"):
        sub = ids[i:i + chunk]
        for attempt in range(5):
            try:
                rr = requests.post(f"{S2_BASE}/paper/batch", headers=HDRS,
                                   params={"fields": fields}, json={"ids": sub}, timeout=40)
                if rr.status_code == 200:
                    for pid, obj in zip(sub, rr.json()):
                        res[pid] = obj
                    break
                elif rr.status_code == 429:
                    time.sleep(2 * (attempt + 1))
                else:
                    time.sleep(1); break
            except Exception:
                time.sleep(1)
        time.sleep(0.4)
    return res

id_list = sorted(papers)
todo = [p for p in id_list if p not in _cache]            # only fetch what's not cached
print(f"{len(id_list)} papers | cached {len(id_list)-len(todo)} | fetching {len(todo)} via S2 batch...")
if todo:
    got = _s2_batch([s2_id(p) for p in todo], "paperId,year,references.paperId", chunk=100)
    for raw in todo:
        obj = got.get(s2_id(raw)) or {}
        _cache[raw] = {"pid":  obj.get("paperId"),
                       "year": obj.get("year"),
                       "refs": [x.get("paperId") for x in (obj.get("references") or []) if x.get("paperId")]}
    CACHE_PATH.write_text(json.dumps(_cache), encoding="utf-8")
    print(f"cached refs for {len(_cache)} papers -> {CACHE_PATH}")

# in-memory graph structures from cache
refs, year, pid_of, resolved = {}, {}, {}, 0
for raw in id_list:
    e = _cache.get(raw) or {}
    if e.get("pid"):
        resolved += 1
        pid_of[raw] = e["pid"]
        year[raw]   = e.get("year")
        refs[raw]   = set(e.get("refs") or []) | {e["pid"]}
print(f"resolved {resolved}/{len(id_list)} papers ({100*resolved/max(1,len(id_list)):.0f}%)")

def cites(a, b):
    return bool(pid_of.get(b) and a in refs and pid_of[b] in refs[a])
def shared_refs(a, b):
    return len(refs.get(a, set()) & refs.get(b, set())) if (a in refs and b in refs) else 0

21156 papers | cached 0 | fetching 21156 via S2 batch...


S2 batch:   0%|          | 0/212 [00:00<?, ?it/s]

cached refs for 21156 papers -> /kaggle/working/citecheck_refs_cache.json
resolved 21116/21156 papers (100%)


In [ ]:
# == 4. verify_relation + run over all tuples =================================
# COCITE_MIN calibration (nb2 human gold, n=83 graph-resolvable relational claims):
#   5-fold CV sweep over tau in {1..10}, selecting by F1 on unsupported-vs-human_label.
#   tau=1 won 4/5 folds; out-of-sample F1=0.442 (P=0.288 R=0.950) vs the uncalibrated
#   default tau=3's in-sample F1=0.437 (P=0.284 R=0.950). Precision is capped ~0.28-0.30
#   regardless of tau because 64% of gold relational claims are builds_on/uses_method_of/
#   supersedes, whose verdict does NOT depend on COCITE_MIN at all (pure citation-edge
#   check) -- co-citation threshold only touches compares/shares_topic. Tested adding a
#   co-citation fallback for the builds_on branch too (accept high shared-refs as
#   "supported" when no direct edge exists): it strictly hurt F1 at every setting tried,
#   so the strict citation-edge rule for builds_on stays as-is. Net effect of the tau
#   change alone: full-universe F1 0.323->0.337, graph-resolvable-subset F1 0.381->0.400,
#   recall stays ~0.94-0.95 on the resolvable subset either way. Precision is a genuine,
#   reported limitation (~70% of flags on the resolvable subset are false positives) --
#   not a tuning artifact.
COCITE_MIN = 1

def verify_relation(a, relation, b):
    if a not in refs or b not in refs:
        return "unverifiable"            # a paper missing from the graph
    a_b, b_a = cites(a, b), cites(b, a)
    sh = shared_refs(a, b)
    ya, yb = year.get(a), year.get(b)
    if relation in ("builds_on", "uses_method_of", "supersedes"):
        if a_b and (ya is None or yb is None or ya >= yb):  # A cites B, A not older
            return "supported"
        if b_a:
            return "supported"           # real citation link (direction may be mislabeled)
        return "unsupported"             # claimed dependency with NO citation edge -> hallucination
    if relation == "compares":
        return "supported" if (a_b or b_a or sh >= COCITE_MIN) else "unsupported"
    if relation == "shares_topic":
        return "supported" if (sh >= COCITE_MIN or a_b or b_a) else "unsupported"
    return "unverifiable"

results = [{"id": iid, "claim_id": cid, "relation": rel,
            "verdict": verify_relation(a, rel, b)} for (iid, cid, a, rel, b) in tuples]

with open(OUT_DIR / "citecheck_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f)
dist = Counter(r["verdict"] for r in results)
cov = sum(1 for r in results if r["verdict"] != "unverifiable") / max(1, len(results))
print("CiteCheck verdicts:", dict(dist))
print(f"graph coverage (adjudicable): {cov:.1%}")
print(f"saved {OUT_DIR / 'citecheck_results.json'}")

In [8]:
# == 5. Per-relation-type breakdown + sanity ==================================
by = defaultdict(Counter)
for (iid, cid, a, rel, b), r in zip(tuples, results):
    by[rel][r["verdict"]] += 1
print("verdict by relation type:")
for rel, c in by.items():
    n = sum(c.values())
    print(f"  {rel:16s} n={n:5d} | " + "  ".join(f"{k}={c[k]}({100*c[k]/n:.0f}%)" for k in ["supported","unsupported","unverifiable"]))
print("\\nInterpretation: 'supported' = citation evidence exists; 'unsupported' = claimed relation")
print("with NO citation link (a relational hallucination CiteCheck catches); 'unverifiable' = paper")
print("missing from the graph. The finding (nb 02) scores these vs HUMAN gold, not vs the graph.")

verdict by relation type:
  builds_on        n= 6183 | supported=666(11%)  unsupported=5503(89%)  unverifiable=14(0%)
  shares_topic     n= 2857 | supported=1159(41%)  unsupported=1693(59%)  unverifiable=5(0%)
  compares         n= 2394 | supported=843(35%)  unsupported=1542(64%)  unverifiable=9(0%)
  supersedes       n=   83 | supported=12(14%)  unsupported=70(84%)  unverifiable=1(1%)
\nInterpretation: 'supported' = citation evidence exists; 'unsupported' = claimed relation
with NO citation link (a relational hallucination CiteCheck catches); 'unverifiable' = paper
missing from the graph. The finding (nb 02) scores these vs HUMAN gold, not vs the graph.


## 6. Next
- Save `citecheck_results.json` (add to your dataset) — `02_relational_finding` consumes it.
- **Calibrate** `COCITE_MIN` and the verdict rules on the human-gold subset (nb2) before reporting.
- **Enhancement (later):** add SciCite citation-*intent* so an edge is read as *builds-on* vs *background* vs *comparison*; add co-citation via a `citations.paperId` fetch.